# Lab 10

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

<img src="cs88-logo.png" alt="CS88 Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

## Lab 10: Iterators and Generators

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab10.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Iterables and Iterators

<hr style="border: 1px solid #fdb515;" />

Remember the `for` loop?  (We really hope so.)  The object the `for` loop
iterates over must be an iterable! An **iterable** is a way of representing
explicit sequences (like lists or strings) as well as *implicit* sequences (like
the natural numbers `1, 2, 3, ...`).

```python
    for elem in iterable:
        # do something
```

`for` loops only work with iterables. This means the object you want to use a
`for` loop on must implement the **iterable interface**.  To implement the
iterable interface, an object must define an `__iter__` method that returns an
object that implements the **iterator interface**.  

To also implement the iterator interface, an object must define a `__next__` 
method to compute and return the next element in the sequence. If the iterator exhausts the sequence, it raises `StopIteration` to send a signal to indicate 
that it reaches the end.

An iterable object can create an arbitrary amount of iterator objects. In
addition, the iterators are independent of each other; in other words they can
have a different position in the sequence.

Here is a table summarizing the required methods of the *iterable* and 
*iterator* interfaces/protocols. Python also has more
[documentation](https://docs.python.org/3/library/stdtypes.html#typeiter)
about [iterator](https://docs.python.org/3/glossary.html#term-iterator) types.

| Iterable | Iterator |
|----------|----------|
| `__iter__` method returns a new iterator | `__iter__` method must return itself |
| | `__next__` method returns the next element, or raises `StopIteration` |

In Python, an iterator must also be an iterable. In other words, it must have a
`__iter__` method that returns itself (with the current state unaltered).

**Analogy**: an iterable is like a book (one can flip through the
pages) and an iterator is a bookmark (saves the position and can locate
the next page).  Calling `__iter__` on a book gives you a new bookmark
independent of other bookmarks, but calling `__iter__` on a bookmark
gives you the bookmark itself, without changing its position at all.

Here is an example of a class definition for an object that implements
the iterator interface:

In [ ]:
class AnIterator:
    def __init__(self):
        self.current = 0

    def __next__(self):
        if self.current > 5:
            raise StopIteration
        self.current += 1
        return self.current

    def __iter__(self):
        return self

Let's go ahead and try out our new toy.

In [ ]:
for num in AnIterator():
    print(num)

This is somewhat equivalent to running:

In [ ]:
t = AnIterator()
t = iter(t) # iter(t) is the same as t.__iter__()
try:
    while True:
        # next(t) is the same as t.__next__()
        print(next(t))
except StopIteration as e:
    pass

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Does it Work?

<hr style="border: 1px solid #fdb515;" />

Consider the following iterators. Which ones work and which ones don't? Why?

**_Note_**: This question isn't graded and you can see the solution by clicking on the dropdown below each Iterator.

In [ ]:
class IteratorA:
    def __init__(self):
        self.start = 10

    def __next__(self):
        if self.start > 100:
            raise StopIteration
        self.start += 20
        return self.start

    def __iter__(self):
        return self

<details style="margin: 10px 0; padding: 10px; border: 1px solid #ccc; border-radius: 5px;">
<summary style="cursor: pointer; font-weight: bold;">IteratorA Solution</summary>

No problem, this is a beautiful iterator.
</details>

In [ ]:
class IteratorB:
    def __init__(self):
        self.start = 5

    def __iter__(self):
        return self

<details style="margin: 10px 0; padding: 10px; border: 1px solid #ccc; border-radius: 5px;">
<summary style="cursor: pointer; font-weight: bold;">IteratorB Solution</summary>

Oh no!  Where is `__next__`?  This fails to implement the iterator
interface because calling `__iter__` doesn't return something that has
a `__next__` method.
</details>

In [ ]:
class IteratorC:
    def __init__(self):
        self.start = 5

    def __next__(self):
        if self.start == 10:
            raise StopIteration
        self.start += 1
        return self.start

<details style="margin: 10px 0; padding: 10px; border: 1px solid #ccc; border-radius: 5px;">
<summary style="cursor: pointer; font-weight: bold;">IteratorC Solution</summary>

This also fails to implement the iterator interface.  Without the
`__iter__` method, the `for` loop will error.  The `for` loop needs to
call `__iter__` first because some objects might not implement the
`__next__` method themselves, but calling `__iter__` will return an
object that does.
</details>

In [ ]:
class IteratorD:
    def __init__(self):
        self.start = 1

    def __next__(self):
        self.start += 1
        return self.start

    def __iter__(self):
        return self

<details style="margin: 10px 0; padding: 10px; border: 1px solid #ccc; border-radius: 5px;">
<summary style="cursor: pointer; font-weight: bold;">IteratorD Solution</summary>

This is technically an iterator, because it implements both `__iter__` and
`__next__`. Notice that it's an infinite sequence!  Sequences like these are
the reason iterators are useful.  Because iterators delay computation, we can
use a finite amount of memory to represent an infinitely long sequence.
</details>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Generators

<hr style="border: 1px solid #fdb515;" />

A generator function returns a special type of **iterator** called
a **generator** object. Such functions can be written using a `yield`
statement:


```python
    def <generator_fn_name>():
        <somevariable> = <something>
        while <predicate>:
            yield <something>
            <increment somevariable>
```

Calling a generator function (a function with a yield statement in it)
makes it return a generator object rather than executing the body of
the function.

The reason we say a generator object is  a special type of **iterator**
is that it has all the properties of an iterator, meaning that:

* Calling the `__iter__` method makes a generator object return
  itself without modifying its current state.
* Calling the `__next__` method makes a generator object compute and
  return the next object in its sequence. If the sequence is
  exhausted, `StopIteration` is raised.
* Typically, a generator should not restart unless it's defined that way. But
  calling the generator function returns a brand new generator object (like
  calling `__iter__` on an iterable object).

However, they do have some fundamental differences:

* An iterator is a class with `__next__` and `__iter__` explicitly defined, but
  a generator can be written as a mere function with a `yield` in it.
* `__iter__` in an iterator uses `return`, but a generator uses `yield`.
* A generator "remembers" its state for the next `__next__` call. Therefore, the
  first `__next__` call works like this:
  1. Enter the function, run until the line with `yield`.
  2. Return the value in the `yield` statement, but remember the state of the
     function for future `__next__` calls.

  And subsequent `__next__` calls work like this:
  1. Re-enter the function, start at **the line after `yield`**, and run until
     the next `yield` statement.
  2. Return the value in the `yield` statement, but remember the state of the
     function for future `__next__` calls.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Final Countdown Part 1

<hr style="border: 1px solid #fdb515;" />

Write a generator function that counts down to 0.

**_Hint_**: Is it possible to not have a `__next__` method in `Countdown`?

In [ ]:
def countdown(n):
    """
    A generator that counts down from N to 0.
    >>> for number in countdown(5):
    ...     print(number)
    ...
    5
    4
    3
    2
    1
    0
    >>> for number in countdown(2):
    ...     print(number)
    ...
    2
    1
    0
    """
    "*** YOUR CODE HERE ***"


In [ ]:
grader.check("countdown")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Final Countdown Part 2

<hr style="border: 1px solid #fdb515;" />

Write a generator that generates prime numbers.
Fill out the `is_prime` helper function and use that 
to create your generator.

In [ ]:
from math import sqrt
def is_prime(n):
    """
    Return True if n is prime, false otherwise.

    >>> is_prime(1)
    False
    >>> is_prime(2)
    True
    >>> is_prime(19)
    True
    """
    "*** YOUR CODE HERE ***"

def primes():
    """
    An infinite generator that outputs primes. 

    >>> p = primes()
    >>> for i in range(3):
    ...     print(next(p))
    ...
    2
    3
    5
    """
    "*** YOUR CODE HERE ***"


In [ ]:
grader.check("primes")